# Regular Expressions - Parsing a Real Data File

Instrument and survey files arrive in whatever layout the instrument writes: mixed tabs and commas, header junk, ragged rows. This notebook turns one such astronomy file into a clean numeric array twice - once with nested `split` calls, once with a single regular expression - and the line count tells the story.

## Learning objectives

- Read a delimited text file whose separators are not consistent.
- Parse it into a numeric array with nested `split` calls.
- Replace that logic with one `re.findall` pattern that matches numbers and identifiers.
- Explain why a character-class pattern survives layout changes that break a chain of splits.

## Background

The pattern used here matches "runs of characters that make up a number or a word", which is the general trick for messy numeric files: rather than describing the separators, describe what you want to keep.

**Prerequisites:** `U3-3_Regex-1_Intro.ipynb`

**Dataset:** a galaxy photometry text file from the instructor's local `Datasets/` folder.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

In [1]:
data_folder = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'

file_path = data_folder + 'Text Processing/587722984435351614_combined.txt'

with open(file_path,'r') as f:
    lines = f.readlines()
# end

lines = lines[:1294]
print(lines[0])

r3a10wnahmplynq9tki_0002,0.9459459459459459,35,37	-9.93853,-4.5805,3.27377,-0.50008,-2.45565,-1.07799,23.33004,24.69427,3.49825,5.32056,309.6923,36.8125,41.78471,51.42857,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.94594,0,0,0.18901,-22.47999,9.02372,0.0,1.0,0.0,0.0,0.0,0.0



## 1. Without regex

In [3]:
data = []
for line in lines:
    x = line.split("\t")[0]
    y = line.split("\t")[1]
    x = x.split(',')[1:]
    y = y.split(',')
    x = x + y
    x = np.array(x).astype(float)
    data.append(x)
# end
data = np.array(data)
data[:2]

array([[ 9.45945946e-01,  3.50000000e+01,  3.70000000e+01,
        -9.93853000e+00, -4.58050000e+00,  3.27377000e+00,
        -5.00080000e-01, -2.45565000e+00, -1.07799000e+00,
         2.33300400e+01,  2.46942700e+01,  3.49825000e+00,
         5.32056000e+00,  3.09692300e+02,  3.68125000e+01,
         4.17847100e+01,  5.14285700e+01,  3.00000000e-01,
         3.00000000e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  9.45940000e-01,  0.00000000e+00,
         0.00000000e+00,  1.89010000e-01, -2.24799900e+01,
         9.02372000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 9.41176471e-01,  3.20000000e+01,  3.40000000e+01,
        -9.93853000e+00, -4.58050000e+00,  2.08441100e+01,
        -1.74126000e+00, -2.76980000e+00,  6.11683000e+00,
         2.70229900e+01,  2.30854200e+01,  3.49716000e+00,
         4.74238000e+00,  2.99

## 2. With regex

In [4]:
data = [ re.findall(r"[\-\w\.]+", line) for line in lines ]
data = np.array(data)[:,1:].astype(float)
data[:2]

array([[ 9.45945946e-01,  3.50000000e+01,  3.70000000e+01,
        -9.93853000e+00, -4.58050000e+00,  3.27377000e+00,
        -5.00080000e-01, -2.45565000e+00, -1.07799000e+00,
         2.33300400e+01,  2.46942700e+01,  3.49825000e+00,
         5.32056000e+00,  3.09692300e+02,  3.68125000e+01,
         4.17847100e+01,  5.14285700e+01,  3.00000000e-01,
         3.00000000e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  9.45940000e-01,  0.00000000e+00,
         0.00000000e+00,  1.89010000e-01, -2.24799900e+01,
         9.02372000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 9.41176471e-01,  3.20000000e+01,  3.40000000e+01,
        -9.93853000e+00, -4.58050000e+00,  2.08441100e+01,
        -1.74126000e+00, -2.76980000e+00,  6.11683000e+00,
         2.70229900e+01,  2.30854200e+01,  3.49716000e+00,
         4.74238000e+00,  2.99

## 3. Review

**Takeaways**

- **Describe what you want, not what separates it.** `split('\t')` followed by `split(',')` hard-codes the file's exact punctuation; a character class that matches numbers keeps working when that punctuation changes.
- **The regex version is shorter and more robust at the same time**, which is unusual enough to be worth noticing - most simplifications cost you something.
- **Neither version validates anything.** Both will happily produce an array of the wrong shape if the file changes, so check the shape after parsing, every time.

**Next ->** `U3-3_Regex-4_ReadElectronData.ipynb` takes on a harder file, where the numbers themselves are written in a format Python does not recognise.